# 🚀 PinokioCloud for Google Colab

**AI App Manager with Dark Mode UI & Live Logs - GitHub Integration**

Welcome to PinokioCloud - your cloud-native Pinokio alternative optimized for Google Colab!

This notebook automatically clones all required scripts from GitHub, making it a **single-file solution**.

## Features:
- ✅ **5+ Verified AI Apps** - Curated database of working Pinokio apps
- 🎨 **Beautiful Dark Mode UI** - Cyberpunk-inspired design with glassmorphism
- 📊 **Live Running Logs** - Real-time color-coded logs in sidebar
- 🎯 **Toast Notifications** - Success/error messages with auto-dismiss
- 🔄 **Full Lifecycle Management** - Install, run, stop, uninstall apps
- 🌐 **Public Access via ngrok** - Share your running apps with the world
- 🎮 **GPU Detection** - Automatic NVIDIA GPU detection and optimization
- 📦 **GitHub Integration** - Auto-clones all scripts from repository
- 🔐 **Authenticated ngrok** - No time limits with hardcoded token

**Repository:** https://github.com/remphanostar/SD-LongNose

---

In [ ]:
#@title 🛠️ Setup: Clone Repository & Install Dependencies
#@markdown **This cell clones the PinokioCloud repository and installs all dependencies**

import os
import sys
import subprocess
import time
from pathlib import Path

print("🚀 Setting up PinokioCloud for Google Colab...")
print("=" * 60)

# Detect environment
if 'google.colab' in sys.modules:
    print("✅ Google Colab environment detected")
    work_dir = Path("/content")
else:
    print("⚠️ Running outside Colab - using current directory")
    work_dir = Path.cwd()

print(f"📁 Working directory: {work_dir}")

# Repository settings
repo_url = "https://github.com/remphanostar/SD-LongNose.git"
repo_dir = work_dir / "SD-LongNose"
colab_dir = repo_dir / "PinokioCloud_Colab"

print(f"🌐 Repository URL: {repo_url}")
print(f"📂 Repository will be cloned to: {repo_dir}")
print(f"🧪 PinokioCloud scripts: {colab_dir}")

# Clone or update repository
if repo_dir.exists():
    print("\n🔄 Repository already exists, updating...")
    try:
        result = subprocess.run(
            ["git", "pull", "origin", "main"],
            cwd=str(repo_dir),
            capture_output=True,
            text=True,
            timeout=60
        )
        if result.returncode == 0:
            print("✅ Repository updated successfully")
        else:
            print(f"⚠️ Git pull failed: {result.stderr}")
            print("🔄 Trying fresh clone...")
            import shutil
            shutil.rmtree(repo_dir)
            subprocess.run(["git", "clone", repo_url, str(repo_dir)], timeout=120)
    except Exception as e:
        print(f"❌ Update failed: {e}")
        print("🔄 Trying fresh clone...")
        import shutil
        shutil.rmtree(repo_dir)
        subprocess.run(["git", "clone", repo_url, str(repo_dir)], timeout=120)
else:
    print("\n⬇️ Cloning repository...")
    try:
        result = subprocess.run(
            ["git", "clone", repo_url, str(repo_dir)],
            capture_output=True,
            text=True,
            timeout=120
        )
        if result.returncode == 0:
            print("✅ Repository cloned successfully")
        else:
            print(f"❌ Git clone failed: {result.stderr}")
            raise Exception("Repository clone failed")
    except Exception as e:
        print(f"❌ Failed to clone repository: {e}")
        raise

# Install dependencies
print("\n📦 Installing Python dependencies...")
dependencies = [
    "streamlit>=1.28.0",
    "gitpython>=3.1.0", 
    "psutil>=5.9.0",
    "pyngrok>=7.0.0",
    "nest_asyncio>=1.5.0",
    "pandas>=1.5.0",
    "requests>=2.28.0"
]

for dep in dependencies:
    print(f"  Installing {dep}...")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", dep], 
                      capture_output=True, check=True, timeout=120)
        print(f"  ✅ {dep} installed")
    except subprocess.CalledProcessError as e:
        print(f"  ❌ Failed to install {dep}: {e}")

# Enable nest_asyncio for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio enabled")

# Verify critical files exist
print("\n🔍 Verifying cloned files...")
critical_files = [
    colab_dir / "streamlit_colab.py",
    colab_dir / "unified_engine.py", 
    colab_dir / "cleaned_pinokio_apps.json"
]

all_files_exist = True
for file_path in critical_files:
    if file_path.exists():
        print(f"  ✅ {file_path.name} - Found ({file_path.stat().st_size} bytes)")
    else:
        print(f"  ❌ {file_path.name} - NOT FOUND")
        all_files_exist = False

if all_files_exist:
    print("\n✅ All required files verified successfully!")
    print("📝 PinokioCloud setup complete")
    print("\n➡️ Continue to the next cell to detect your GPU")
else:
    print("\n❌ Required files missing from repository")
    raise FileNotFoundError("PinokioCloud files not found in repository")

# Add repository to Python path
if str(colab_dir) not in sys.path:
    sys.path.insert(0, str(colab_dir))
    print(f"🐍 Added {colab_dir} to Python path")

print("\n" + "=" * 60)
print("🎉 Setup completed successfully!")
print("=" * 60)

In [ ]:
#@title 🎮 GPU Detection & System Information
#@markdown **Detect available GPU hardware and system capabilities**

import subprocess
import platform
import sys

def detect_gpu():
    """Detect GPU information using multiple methods"""
    print("🔍 Detecting GPU hardware...\n")
    
    # Method 1: nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,memory.free', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=10
        )
        
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split('\n')
            print("🎮 NVIDIA GPU(s) detected:")
            for i, gpu in enumerate(gpu_info):
                name, total, used, free = gpu.split(', ')
                print(f"  GPU {i}: {name}")
                print(f"    Total VRAM: {total} MB")
                print(f"    Used VRAM: {used} MB")
                print(f"    Free VRAM: {free} MB")
            return True
        else:
            print("❌ nvidia-smi failed or not available")
    except (subprocess.TimeoutExpired, FileNotFoundError) as e:
        print(f"❌ nvidia-smi error: {e}")
    
    # Method 2: PyTorch CUDA detection
    try:
        import torch
        if torch.cuda.is_available():
            print("🎮 PyTorch CUDA detected:")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
            return True
    except ImportError:
        pass
    
    # Method 3: TensorFlow GPU detection
    try:
        import tensorflow as tf
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            print("🎮 TensorFlow GPU detected:")
            for i, gpu in enumerate(gpus):
                print(f"  GPU {i}: {gpu.name}")
            return True
    except ImportError:
        pass
    
    print("⚠️ No GPU detected - will run in CPU mode")
    return False

def get_system_info():
    """Get system information"""
    print("\n🖥️ System Information:")
    print(f"  Platform: {platform.system()} {platform.release()}")
    print(f"  Architecture: {platform.machine()}")
    print(f"  Python: {sys.version.split()[0]}")
    
    # Check if running in Colab
    if 'google.colab' in sys.modules:
        print(f"  Environment: Google Colab")
    else:
        print(f"  Environment: Local/Other")
    
    # Memory info
    try:
        import psutil
        memory = psutil.virtual_memory()
        print(f"  RAM: {memory.total // (1024**3)} GB total, {memory.available // (1024**3)} GB available")
    except ImportError:
        print("  RAM: Could not detect (psutil not available)")

# Run detection
gpu_available = detect_gpu()
get_system_info()

if gpu_available:
    print("\n✅ Your system is ready for GPU-accelerated AI apps!")
else:
    print("\n⚠️ No GPU detected - CPU-only mode will be used")

print("\n➡️ Continue to the next cell to launch PinokioCloud")

In [ ]:
#@title 🚀 Launch PinokioCloud with ngrok Tunnel (Authenticated)
#@markdown **Launch the beautiful Streamlit UI with public access via ngrok**

import subprocess
import threading
import time
import os
import sys
from pathlib import Path

# Configuration
STREAMLIT_PORT = 8501
NGROK_REGION = "us"  #@param ["us", "eu", "ap", "au", "sa", "jp", "in"]

# Define paths from repository
WORK_DIR = Path("/content") if 'google.colab' in sys.modules else Path.cwd()
REPO_DIR = WORK_DIR / "SD-LongNose"
COLAB_DIR = REPO_DIR / "PinokioCloud_Colab"
STREAMLIT_FILE = COLAB_DIR / "streamlit_colab.py"

def setup_ngrok():
    """Set up ngrok tunnel with authentication"""
    try:
        from pyngrok import ngrok, conf
        
        # Set ngrok authentication token
        ngrok.set_auth_token("31hXIA9IPUsFAnKFrZ92A7jgwj3_5zwfYZfiJRYGacXizWs9K")
        
        # Set ngrok region
        conf.get_default().region = NGROK_REGION
        
        # Create tunnel
        print(f"🌐 Creating authenticated ngrok tunnel on port {STREAMLIT_PORT}...")
        public_tunnel = ngrok.connect(STREAMLIT_PORT)
        
        print(f"✅ Public URL: {public_tunnel.public_url}")
        print(f"🌍 Share this URL to access PinokioCloud from anywhere!")
        print(f"🔐 Using authenticated ngrok (no time limits)")
        
        return public_tunnel.public_url
        
    except Exception as e:
        print(f"❌ Failed to create ngrok tunnel: {e}")
        return None

def launch_streamlit():
    """Launch Streamlit application from cloned repository"""
    try:
        # Verify streamlit file exists
        if not STREAMLIT_FILE.exists():
            print(f"❌ Streamlit file not found: {STREAMLIT_FILE}")
            print(f"🔍 Repository directory contents:")
            if REPO_DIR.exists():
                for item in REPO_DIR.iterdir():
                    print(f"  - {item}")
            return None
        
        # Change to repository directory
        original_cwd = os.getcwd()
        os.chdir(str(COLAB_DIR))
        print(f"📁 Changed directory to: {COLAB_DIR}")
        
        print(f"🚀 Launching Streamlit from: {STREAMLIT_FILE}")
        
        # Launch Streamlit
        cmd = [
            sys.executable, "-m", "streamlit", "run", str(STREAMLIT_FILE),
            "--server.port", str(STREAMLIT_PORT),
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
            "--browser.gatherUsageStats", "false"
        ]
        
        print(f"📋 Command: {' '.join(cmd)}")
        
        # Start Streamlit in background
        process = subprocess.Popen(
            cmd, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            cwd=str(COLAB_DIR)
        )
        
        # Wait a moment for Streamlit to start
        print("⏳ Starting Streamlit server...")
        time.sleep(10)
        
        # Check if process is running
        if process.poll() is None:
            print("✅ Streamlit server started successfully!")
            return process
        else:
            print("❌ Streamlit server failed to start")
            # Print any error output
            output, _ = process.communicate()
            print(f"Error output: {output}")
            return None
        
    except Exception as e:
        print(f"❌ Failed to launch Streamlit: {e}")
        return None
    finally:
        # Restore original directory
        try:
            os.chdir(original_cwd)
        except:
            pass

# Verify repository is available
if not REPO_DIR.exists():
    print("❌ Repository not found! Please run the setup cell first.")
    print(f"Expected location: {REPO_DIR}")
else:
    print(f"✅ Repository found: {REPO_DIR}")
    print(f"✅ PinokioCloud scripts: {COLAB_DIR}")
    
    # Launch the application
    print("\n🚀 Launching PinokioCloud from GitHub repository...")
    print("=" * 60)
    
    # Start Streamlit
    streamlit_process = launch_streamlit()
    
    if streamlit_process:
        # Set up ngrok tunnel
        public_url = setup_ngrok()
        
        if public_url:
            print("\n" + "=" * 60)
            print("🎉 PinokioCloud is now running from GitHub!")
            print("=" * 60)
            print(f"🌐 Public URL: {public_url}")
            print(f"📱 Local URL: http://localhost:{STREAMLIT_PORT}")
            print(f"📦 Repository: https://github.com/remphanostar/SD-LongNose")
            print("=" * 60)
            print("\n🎯 Features available:")
            print("  • Browse 5+ verified AI apps")
            print("  • Install apps with one click")
            print("  • Run apps with GPU acceleration")
            print("  • Live logs and toast notifications")
            print("  • Beautiful dark mode UI")
            print("  • All scripts loaded from GitHub")
            print("  • 🔐 Authenticated ngrok (no time limits)")
            print("\n⚠️ Keep this cell running to maintain the server")
            print("\n🔄 To stop the server, interrupt this cell or restart the runtime")
            
            # Keep the process running
            try:
                streamlit_process.wait()
            except KeyboardInterrupt:
                print("\n🛑 Stopping PinokioCloud...")
                streamlit_process.terminate()
                print("✅ Server stopped")
        else:
            print("⚠️ Running without public tunnel - only local access available")
            print(f"📱 Local URL: http://localhost:{STREAMLIT_PORT}")
    else:
        print("❌ Failed to start PinokioCloud")
        print("\n🔧 Troubleshooting:")
        print("  1. Make sure the repository was cloned correctly")
        print("  2. Check that dependencies were installed successfully") 
        print("  3. Try restarting the runtime and running all cells again")
        print(f"  4. Verify files exist in: {COLAB_DIR}")

# 📖 Usage Instructions

## 🎯 Getting Started
1. **Run all cells above** in order to set up and launch PinokioCloud
2. **All scripts are automatically cloned** from the GitHub repository
3. **Click the public ngrok URL** to access the beautiful web interface
4. **Browse the app library** to discover 5+ verified AI applications
5. **Install and run apps** with one-click functionality

## 🎨 Interface Features
- **🌙 Dark Mode UI**: Beautiful cyberpunk-inspired design with glassmorphism effects
- **📊 Live Logs**: Real-time color-coded logs displayed in the sidebar
- **🎯 Toast Notifications**: Success/error messages with automatic dismissal
- **🔄 Manual Refresh**: Refresh logs manually to prevent constant flickering

## 🔧 App Management
- **Install**: Clone repositories and run installation scripts automatically
- **Run**: Execute start scripts and launch web interfaces  
- **Stop**: Terminate running processes safely
- **Uninstall**: Complete removal including files and virtual environments

## 🌐 Public Access
- Your PinokioCloud instance is accessible via the **authenticated ngrok public URL**
- Share this URL with others to give them access to your AI apps
- Apps running on your instance will also be publicly accessible
- **No time limits** with authenticated ngrok token

## ⚠️ Important Notes
- **Keep the launch cell running** to maintain the server
- **GPU acceleration** is automatically used when available
- **Storage is ephemeral** - installed apps will be lost when runtime restarts
- **Resource limits** apply based on your Colab plan
- **GitHub dependency** - requires internet connection to clone repository
- **Authenticated ngrok** - uses hardcoded token for persistent access

---

## 🛠️ Troubleshooting

### Common Issues:
1. **Repository not found**: Re-run the setup cell to clone the repository
2. **Dependencies missing**: Restart runtime and run setup cell again
3. **Streamlit won't start**: Check that all files exist in the repository
4. **ngrok tunnel fails**: Verify internet connection and ngrok token
5. **Apps won't install**: Check GPU availability and storage space

### Support:
- **Repository**: https://github.com/remphanostar/SD-LongNose
- **Issues**: Report problems via GitHub Issues
- **Updates**: Pull latest changes by re-running setup cell

---

**Enjoy using PinokioCloud with GitHub integration and authenticated ngrok! 🚀**